# Analyze forces, frequency spectrum and convergence 

In [ ]:
import matplotlib.pyplot as plt

from os import makedirs
from os.path import join, exists
from torch import where, tensor, argmax, from_numpy

from utils import load_yplus, load_ratio_rans_les, load_force_coeffs, interpolate_uniform, compute_fft, compute_norm_of_fields

In [ ]:
# flow quantities
# u_inf = 238.845

# validation @ Ma = 0.73
u_inf = 242.16629

# sweep
# u_inf = 275.795 

# chord length
chord = 0.15

# start of quasi-steady state -> chosen based on course of ratio RANS / LES
tstar_start = 50
tstar_end = 140

# use latex fonts
plt.rcParams.update({"text.usetex": True, "figure.dpi": 360, "text.latex.preamble": r"\usepackage{xcolor}"})

In [ ]:
# define load and save path
load_dir = join("/media", "janis", "Elements", "Janis", "2D_buffet_simulation", "DDES_3D_Ma0.72_Re2e6_AoA5deg")

# URANS with sweep
# load_dir = join("/media", "janis", "Elements1", "Janis", "2D_buffet_simulation", "URANS_3D_Ma0.72_Re2e6_AoA5deg_beta30deg")
# save_dir = join("..", "run", "plots", "URANS_withSweep", "first_test_beta30deg_Ma_normal0.72")
# cases = ["URANS_y25_ymax0.0315_beta30_Ma_normal0.72"]
# legend = [r"k-$\omega$-SST-URANS, $\alpha = 5^\circ, \beta = 30^\circ, \Lambda = 0.21, M_{\infty, n} = 0.72$"]

# URANS with pitching
load_dir = join("/media", "janis", "Elements", "Janis", "2D_buffet_simulation", "URANS_2D_Ma0.72_Re2e6_AoA5deg")
save_dir = join("..", "run", "plots", "URANS_with_pitching")
cases = ["URANS_kOmegaSST_k8.557_omega35.605", "URANS_deltaAoA_2deg"]
legend = [r"k-$\omega$-SST-URANS", r"k-$\omega$-SST-URANS, $\Delta \alpha = 2^\circ$"]


# validation 
load_dir = join("/media", "janis", "Elements", "Janis", "2D_buffet_simulation", "URANS_2D_Ma0.73_Re3e6")
# save_dir = join("..", "run", "plots", "URANS_validation", "comparison_AoA")  # "AoA3.5deg")
# cases = ["URANS_kOmegaSST_alpha2.5deg", "URANS_kOmegaSST_alpha3.5deg"]
# legend = [r"k-$\omega$-SST-URANS, $\alpha = 2.5^\circ$", r"k-$\omega$-SST-URANS, $\alpha = 3.5^\circ$"]

load_dir = join("..", "run", "URANS")
deg = 3.5
save_dir = join("..", "run", "plots", "URANS_validation", "grid_convergence_study", f"AoA{deg}deg")
cases = [f"URANS_kOmegaSST_alpha{deg}deg_coarseGrid", f"URANS_kOmegaSST_alpha{deg}deg",
         f"URANS_kOmegaSST_alpha{deg}deg_fineGrid", f"URANS_kOmegaSST_alpha{deg}deg_veryFineGrid"]
legend = [r"$\alpha = 2.5^\circ$ (coarse grid)", r"$\alpha = 2.5^\circ$ (default grid)", r"$\alpha = 2.5^\circ$ (fine grid)", 
          r"$\alpha = 2.5^\circ$ (very fine grid)"]
legend = [r"$\alpha = 3.5^\circ$ (coarse grid)", r"$\alpha = 3.5^\circ$ (default grid)", r"$\alpha = 3.5^\circ$ (fine grid)", 
          r"$\alpha = 3.5^\circ$ (very fine grid)"]

# create plot directory
if not exists(save_dir):
    makedirs(save_dir)

In [ ]:
# plot ratio RANS / LES vs. time
ratio = [load_ratio_rans_les(join(load_dir, c)) for c in cases]

ls = ["-"] * len(ratio)
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
for i in range(len(ratio)):
    ax.plot(ratio[i].t * u_inf / chord, ratio[i].rans / ratio[i].les, label=legend[i], ls=ls[i])
        
ax.set_xlabel(r"$t \frac{U_{\infty}}{c}$")
ax.set_ylabel("$RANS / LES$")
ax.set_yscale("log")

# mark beginning of investigated time window
ax.axvline(tstar_start, ls=":", color="red")
# ax.set_xlim(ratio[0].t.iloc[0] * u_inf / chord, ratio[0].t.iloc[-1] * u_inf / chord)
ax.set_xlim(0, tstar_end)
fig.tight_layout()
ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")
fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.81)
plt.savefig(join(save_dir, "comparison_RANS_LES.png"))
plt.show()

In [ ]:
# check forces 
forces = [load_force_coeffs(join(load_dir, c)) for c in cases]

ls = ["-"] * len(forces)

# use default color cycle
color = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f"]

fig, ax = plt.subplots(2, 1, figsize=(6, 4), sharex="col")
for i in range(len(cases)):
    ax[0].plot(forces[i].t * u_inf / chord, forces[i].cx, label=legend[i], ls=ls[i], color=color[i])
    ax[1].plot(forces[i].t * u_inf / chord, forces[i].cy, ls=ls[i], color=color[i])

    # print mean cl
    idx_start = where(abs(tensor(f["t"].values) * u_inf / chord - tstar_start) < 1e-3)[0][0].item()
    idx_end = 1000
    try:
        idx_end = where(abs(tensor(f["t"].values) * u_inf / chord - tstar_end) < 1e-3)[0][0].item()
    except:
        pass
    if idx_end == 1000:
        print(f"Mean cl case {i}:\t{round(forces[i].cy[idx_start:].mean(), 3)}")
    else:
        print(f"Mean cl case {i}:\t{round(forces[i].cy[idx_start:idx_end+1].mean(), 3)}")

    # mark mean
    # ax[1].axhline(forces[i].cy[idx_start:].mean(), ls="-.", color=color[i], lw=1)

ax[0].set_ylim(0, 0.038)
ax[1].set_ylim(0.65, 1.38)
ax[0].set_ylabel(r"$c_d$")
ax[1].set_ylabel(r"$c_l$")
ax[-1].set_xlabel(r"$t \frac{U_{\infty}}{c}$")
# ax[-1].set_xlim(0, forces[0].t.iloc[-1] * u_inf / chord)
ax[-1].set_xlim(0, tstar_end)
ax[-1].set_xlim(0, 742)

fig.tight_layout()
for i in range(2):
    ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
    ax[i].minorticks_on()
    ax[i].tick_params(axis="x", which="minor", bottom=False)
    ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")
    
    # mark beginning of investigated time window
    # ax[i].axvline(101.71, ls=":", color="black", zorder=1, lw=1)
    # ax[i].axvline(93.6, ls=":", color="black", zorder=1, lw=1)
    # ax[i].axvline(tstar_start, ls=":", color="red")

# mark min. & max. values for cl of Johannes paper
# ax[1].axhspan(0.8, 1.1, 0, 1, color="black", alpha=0.3, label="$literature$", zorder=0)

# mark min. & max. and mean values for cl of Giannelis et al (Ma = 0.73, Re = 3e6, alpha = 3.5deg), approx.
# ax[1].axhspan(0.81, 0.97, 0, 1, color="black", alpha=0.3, label="$literature$", zorder=0)
# ax[1].axhline(0.89, ls="-.", color="black", zorder=1, lw=1, label=r"$\bar{c}_L$")

fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.86)
# plt.savefig(join(save_dir, "comparison_coefficients_vs_t_mark_cp_time_steps.png"))
# plt.savefig(join(save_dir, "comparison_coefficients_vs_t_comparison_literature.png"))
plt.savefig(join(save_dir, "comparison_coefficients_vs_t_full.png"))
plt.show()

In [ ]:
# plot max y+ vs. time
y_plus = [load_yplus(join(load_dir, c)) for c in cases]

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
for i in range(len(cases)):
    ax.plot(y_plus[i].t * u_inf / chord, y_plus[i].yPlus_max, label=legend[i], ls=ls[i])
        
ax.set_xlabel(r"$t \frac{U_{\infty}}{c}$")
ax.set_ylabel("$y^+_{max}$")
# ax.set_xlim(y_plus[0].t.iloc[0] * u_inf / chord, y_plus[0].t.iloc[-1] * u_inf / chord)
ax.set_xlim(0, tstar_end)
ax.set_ylim(0.5, 1.35)

# mark beginning of investigated time window
ax.axvline(tstar_start, ls=":", color="red")
fig.tight_layout()
ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")
fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.82)
plt.savefig(join(save_dir, "comparison_yPlus_max_vs_t.png"))
plt.show()

In [ ]:
# plot avg. y+ vs. time
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
for i in range(len(cases)):
    ax.plot(y_plus[i].t * u_inf / chord, y_plus[i].yPlus_avg, label=legend[i], ls=ls[i])
        
ax.set_xlabel(r"$t \frac{U_{\infty}}{c}$")
ax.set_ylabel("$y^+_{avg}$")
# ax.set_xlim(y_plus[0].t.iloc[0] * u_inf / chord, y_plus[0].t.iloc[-1] * u_inf / chord)
ax.set_xlim(0, tstar_end)
ax.set_ylim(0.2, 0.52)

# mark beginning of investigated time window
ax.axvline(tstar_start, ls=":", color="red", zorder=0)
fig.tight_layout()
ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")
fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.82)
plt.savefig(join(save_dir, "comparison_yPlus_avg_vs_t.png"))
plt.show()

In [ ]:
# re-sample the force coefficients since the simulation is run with a variable time step
t_uniform, cy_inter = [], []
for f in forces:
    idx_start = where(abs(tensor(f["t"].values) * u_inf / chord - tstar_start) < 1e-3)[0][0].item()

    # remove the transient phase and duplicates (resulting from dt < write precision)
    f.drop(list(range(0, idx_start)), inplace=True)
    
    try:
        f.reset_index(inplace=True, drop=True)
        idx_end = where(abs(tensor(f["t"].values) * u_inf / chord - tstar_end) < 1e-3)[0][0].item()
        f.drop(list(range(idx_end, f.index[-1]+1)), inplace=True)
    except:
        pass
        
    f.reset_index(inplace=True, drop=True)

    # interpolate values
    _t_uniform, _cy_inter = interpolate_uniform(f["t"].values, f["cy"].values)
    t_uniform.append(_t_uniform)
    cy_inter.append(_cy_inter)

In [ ]:
# verify interpolation for all cases
for i, c in enumerate(cases):
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(forces[i]["t"] * u_inf / chord, forces[i]["cy"], label="$original$")
    ax.plot(t_uniform[i] * u_inf / chord, cy_inter[i], ls="--", label="$interpolated$")
    ax.set_xlabel(r"$t \frac{U_{\infty}}{c}$")
    ax.set_ylabel(r"$c_l$")
    ax.set_xlim(t_uniform[i][0] * u_inf / chord, t_uniform[i][-1] * u_inf / chord)
    ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
    ax.minorticks_on()
    ax.tick_params(axis="x", which="minor", bottom=False)
    ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")
    fig.tight_layout()
    fig.legend(ncols=2, loc="upper center")
    fig.subplots_adjust(top=0.88)
    plt.savefig(join(save_dir, f"check_interpolation_{c.split("/")[-1]}.png"))
    plt.show()

In [ ]:
# compute FFT
results_fft_cl = [compute_fft(cy_inter[c], (t_uniform[c][1] - t_uniform[c][0]).item()) for c in range(len(cases))]

idx = [argmax(from_numpy(res[1])) for res in results_fft_cl]

# print out buffet frequencies for each case
for i, res in enumerate(results_fft_cl):
    print(f"Max. amplitude at f = {round(res[0][idx[i]].item(), 5)} Hz (Sr = {round(res[0][idx[i]].item() * chord / u_inf, 5)})")

In [ ]:
# plot frequency of max. amplitude as Sr and Hz
f_legend, sr_legend = [], []
color = ['blue', 'orange', 'green', 'red', 'purple', 'brown', 'pink', 'gray']

fig, ax = plt.subplots(2, 1, figsize=(6, 5), sharey="col")
for i, fa in enumerate(results_fft_cl):
    ax[0].plot(fa[0], fa[1] / max(fa[1]), label=legend[i])
    ax[1].plot(fa[0] * chord / u_inf, fa[1] / max(fa[1]))
    
    # not working -> remains black
    f_legend.append(r"$\textcolor{" + color[i] + "}{f = " + "{:.2f}".format(fa[0][idx[i]].item()) + r"\, Hz" + "}$")
    sr_legend.append(r"$\textcolor{" + color[i] + "}{Sr = " + "{:.4f}".format(fa[0][idx[i]].item() * chord / u_inf) + "}$")

ax[0].set_xscale("log")
ax[0].set_xlim(10, 1000)
ax[1].set_xlim(0.02, 0.14)
ax[0].set_xlabel("$f$ $[Hz]$")
ax[1].set_xlabel("$Sr$ $[-]$")

# TODO: rendering of colors for annotations and placement not working for some reason
h_pos = 0.5
# print(f_legend)
# ax[0].annotate(r"$\textcolor{blue}{bla}$", (15, 5e-4), ha="left", bbox=dict(facecolor="white", edgecolor="none"))
ax[0].annotate(r"\newline".join(f_legend), (300, h_pos), ha="left", bbox=dict(facecolor="white", edgecolor="none"))
ax[1].annotate(r"\\".join(sr_legend), (0.11, h_pos), ha="left", bbox=dict(facecolor="white", edgecolor="none"))

for i in range(2):
    ax[i].set_ylabel("$A / A_{max}$")
    ax[i].ticklabel_format(style="sci", axis="y", scilimits=(0, 0))

# mark Sr of literature
ax[0].axvspan(0.056 * u_inf / chord, 0.076 * u_inf / chord, 0, 1, color="black", alpha=0.3, label="$literature$", zorder=0)
ax[1].axvspan(0.056, 0.076, 0, 1, color="black", alpha=0.3, zorder=0)

fig.tight_layout()
fig.legend(ncols=2, loc="upper center")
fig.subplots_adjust(top=0.84)
plt.savefig(join(save_dir, "dominant_frequency.png"))
plt.show()

In [ ]:
# plot complete FFT
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
for i, res in enumerate(results_fft_cl):
    ax.plot(res[0], res[1], label=legend[i])
ax.set_yscale("log")
ax.set_xscale("log")
ax.set_xlabel(r"$f$")
ax.set_ylabel("PSD")
ax.set_ylim(1e-10, 5e-3)
ax.set_xlim(1e1, 1e4)
fig.tight_layout()
ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="x")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="x")
fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.82)
plt.savefig(join(save_dir, "results_fft.png"))
plt.show()

In [ ]:
# compute the convergence of UMean wrt time
results = [compute_norm_of_fields(join(load_dir, c)) for c in cases]

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
i = 0
for times, res in results:
    ax.plot(times * u_inf / chord, res, marker="x", label=legend[i])
    i += 1
ax.set_xlim(results[0][0][0] * u_inf / chord, results[0][0][-1] * u_inf / chord)
ax.set_yscale("log")
ax.set_xlabel(r"$t \frac{U_{\infty}}{d}$")
ax.set_ylabel(r"$\left| \left| \frac{\overline{U}_{t+1} - \overline{U}_{t}}{\Delta t}\right| \right|_F \times "
              r"\frac{1}{||\overline{U}_{t_0} ||_F}$")
ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")
fig.tight_layout()
fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.82)
plt.savefig(join(save_dir, f"convergence_Umean_field_vs_time.png"))
plt.show()

In [ ]:
from torch import linspace, pi, sin
frequency = 90
amplitude = 1
phase = pi/2
alpha = 5.0
t = linspace(0, 0.022, 1000)

AoA = amplitude * sin(2 * pi * frequency * t - phase) + alpha

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
ax.plot(t*u_inf/chord, AoA)
ax.set_xlabel(r"$t \frac{U_{\infty}}{c}$")
ax.set_ylabel(r"$\alpha$")
ax.set_xlim(0, tstar_end)
fig.tight_layout()
plt.show()

In [ ]:
forces = [load_force_coeffs(join(load_dir, c)) for c in cases]

ls = ["-"] * len(forces)
fig, ax = plt.subplots(1, 1, figsize=(6, 3), sharex="col")
for i in range(len(cases)):
    ax.plot(forces[i].t * u_inf / chord, forces[i].cy, ls=ls[i])

# ax[0].set_ylim(0.035, 0.12)
#ax.set_ylim(0.6, 1.3)
ax.set_ylabel(r"$c_l$")
ax.set_xlabel(r"$t \frac{U_{\infty}}{c}$")
# ax[-1].set_xlim(0, forces[0].t.iloc[-1] * u_inf / chord)
ax.set_xlim(0, 100)

fig.tight_layout()
ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")
    
# for j in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0]:
#     ax.axvline(j*10, ls=":", color="red", zorder=1)
ax.axvline(60, ls=":", color="red", zorder=1)
# mark beginning of investigated time window
# ax.axvline(0.015 * u_inf / chord, ls=":", color="red", zorder=1)
# ax.axvline(0.005 * u_inf / chord, ls=":", color="red", zorder=1)

# mark min. & max. values for cl of Johannes paper
# ax[1].axhspan(0.8, 1.1, 0, 1, color="black", alpha=0.3, label="$literature$", zorder=0)

# fig.legend(ncol=2, loc="upper center")
# fig.subplots_adjust(top=0.9)
plt.savefig(join(save_dir, "cl_only_1.png"))
plt.show()